In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from arch import arch_model
import plotly.graph_objects as go
from plotly.subplots import make_subplots

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 1. Модели и датасеты

In [2]:
# === Datasets ===
class VolatilityDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class GatingDataset(Dataset):
    def __init__(self, X, y_real, y_garch):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y_real = torch.FloatTensor(y_real)
        self.y_garch = torch.FloatTensor(y_garch)
    def __len__(self): return len(self.y_real)
    def __getitem__(self, idx): return self.X[idx], self.y_real[idx], self.y_garch[idx]

# === LSTM ===
class VolatilityLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=256, num_layers=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.batch_norm = nn.BatchNorm1d(hidden_size)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.batch_norm(out[:, -1, :])
        out = self.dropout(self.relu(self.fc1(out)))
        return self.fc2(out).squeeze(-1)

# === FG-GINN ===
class FeatureGatingGINN(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=3, dropout=0.2):
        super().__init__()
        self.data_lstm = nn.LSTM(input_size, hidden_size, num_layers,
                                 batch_first=True, dropout=dropout)
        self.data_bn = nn.BatchNorm1d(hidden_size)
        self.garch_proj = nn.Sequential(
            nn.Linear(1, hidden_size // 2), nn.ReLU(),
            nn.Linear(hidden_size // 2, hidden_size))
        self.lambda_lstm = nn.LSTM(input_size, 32, 2, batch_first=True, bidirectional=True)
        self.lambda_bn = nn.BatchNorm1d(64)
        self.lambda_head = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
        self.output_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_size // 2, 1))
    def forward(self, x, garch_signal):
        out, _ = self.data_lstm(x)
        data_feat = self.data_bn(out[:, -1, :])
        garch_feat = self.garch_proj(garch_signal)
        l_out, _ = self.lambda_lstm(x)
        l_feat = self.lambda_bn(l_out[:, -1, :])
        lambda_val = self.lambda_head(l_feat)
        combined = data_feat + lambda_val * garch_feat
        output = self.output_head(combined).squeeze(-1)
        return output, lambda_val.squeeze(-1)

## 2. Универсальные функции

In [3]:
def prepare_all_data(returns, dates, WINDOW=90):
    """
    Полная подготовка данных: GT variance + GARCH rolling + нормализация + split.
    Возвращает словарь со всем необходимым.
    """
    # 1. Ground truth variance
    rolling_mean = returns.rolling(WINDOW).mean()
    raw_var = (returns - rolling_mean) ** 2
    gt_var_raw = raw_var.rolling(5).mean()
    
    # Маска валидных значений
    valid_mask = gt_var_raw.notna()
    gt_var = gt_var_raw[valid_mask].values
    gt_var_log = np.log1p(gt_var)
    
    # Даты — берём по той же маске
    # dates может быть numpy array или pandas, обрабатываем оба случая
    if isinstance(dates, np.ndarray):
        # returns.index может не совпадать с позициями в dates
        # Берём позиции где valid_mask = True
        valid_positions = np.where(valid_mask.values)[0]
        # returns начинается с index 1 (после dropna от diff), dates тоже
        gt_dates = dates[valid_positions]
    else:
        gt_dates = dates[valid_mask].values
    
    valid_idx = gt_var_raw.dropna().index
    
    # 2. GARCH rolling forecast (на тех же индексах что и gt_var)
    print(f'  GARCH rolling на {len(gt_var)} точках...')
    garch_preds = []
    for i, idx in enumerate(valid_idx):
        pos = returns.index.get_loc(idx)
        train_data = returns.iloc[max(0, pos - WINDOW) : pos]
        if len(train_data) < 30:
            garch_preds.append(np.nan)
            continue
        try:
            m = arch_model(train_data, vol='Garch', p=1, q=1, dist='Normal')
            r = m.fit(disp='off')
            f = r.forecast(horizon=1)
            garch_preds.append(f.variance.values[-1, 0])
        except:
            garch_preds.append(np.nan)
        if (i + 1) % 500 == 0:
            print(f'    {i+1}/{len(valid_idx)}')
    
    garch_preds = np.array(garch_preds)
    garch_preds[np.isnan(garch_preds)] = np.nanmean(garch_preds)
    print(f'  GARCH готов!')
    
    # 3. Окна
    X_all, y_all, d_all, g_all, gt_raw_all = [], [], [], [], []
    for i in range(WINDOW, len(gt_var_log)):
        X_all.append(gt_var_log[i - WINDOW : i])
        y_all.append(gt_var_log[i])
        d_all.append(gt_dates[i] if hasattr(gt_dates, '__getitem__') else gt_dates.iloc[i])
        g_all.append(garch_preds[i])  # GARCH для этого же дня
        gt_raw_all.append(gt_var[i])  # сырой gt_var для метрик
    
    X_all = np.array(X_all)
    y_all = np.array(y_all)
    d_all = np.array(d_all)
    g_all = np.array(g_all)
    gt_raw_all = np.array(gt_raw_all)
    
    # 4. Split
    d_dt = d_all.astype('datetime64')
    train_idx = d_dt < np.datetime64('2020-01-01')
    val_idx = (d_dt >= np.datetime64('2020-01-01')) & (d_dt < np.datetime64('2022-01-01'))
    test_idx = d_dt >= np.datetime64('2022-01-01')
    
    # 5. Нормализация X и y
    tr_mean = X_all[train_idx].mean()
    tr_std = X_all[train_idx].std()
    
    X_tr = (X_all[train_idx] - tr_mean) / tr_std
    X_va = (X_all[val_idx] - tr_mean) / tr_std
    X_te = (X_all[test_idx] - tr_mean) / tr_std
    y_tr = (y_all[train_idx] - tr_mean) / tr_std
    y_va = (y_all[val_idx] - tr_mean) / tr_std
    y_te = (y_all[test_idx] - tr_mean) / tr_std
    
    # 6. Нормализация GARCH (log + тот же scaler)
    g_log = np.log1p(g_all)
    g_tr = (g_log[train_idx] - tr_mean) / tr_std
    g_va = (g_log[val_idx] - tr_mean) / tr_std
    g_te = (g_log[test_idx] - tr_mean) / tr_std
    
    print(f'  Train: {train_idx.sum()}, Val: {val_idx.sum()}, Test: {test_idx.sum()}')
    
    return {
        'X_tr': X_tr, 'y_tr': y_tr, 'X_va': X_va, 'y_va': y_va,
        'X_te': X_te, 'y_te': y_te,
        'g_tr': g_tr, 'g_va': g_va, 'g_te': g_te,
        'dates_te': d_all[test_idx],
        'garch_te_raw': g_all[test_idx],  # сырые GARCH для метрик
        'gt_te_raw': gt_raw_all[test_idx],  # сырые gt_var для метрик
        'tr_mean': tr_mean, 'tr_std': tr_std,
    }


def make_loaders(d, batch_size=64):
    """Создаёт DataLoader'ы из словаря данных."""
    lstm_tr = DataLoader(VolatilityDataset(d['X_tr'], d['y_tr']), batch_size, shuffle=True)
    lstm_va = DataLoader(VolatilityDataset(d['X_va'], d['y_va']), batch_size)
    lstm_te = DataLoader(VolatilityDataset(d['X_te'], d['y_te']), batch_size)
    ginn_tr = DataLoader(GatingDataset(d['X_tr'], d['y_tr'], d['g_tr']), batch_size, shuffle=True)
    ginn_va = DataLoader(GatingDataset(d['X_va'], d['y_va'], d['g_va']), batch_size)
    ginn_te = DataLoader(GatingDataset(d['X_te'], d['y_te'], d['g_te']), batch_size)
    return lstm_tr, lstm_va, lstm_te, ginn_tr, ginn_va, ginn_te


def train_lstm(model, train_ld, val_ld, epochs=300, patience=50, name='LSTM'):
    """Обучение LSTM."""
    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=20, factor=0.5)
    mse = nn.MSELoss()
    best_v, best_s, pat = float('inf'), None, 0
    for ep in range(epochs):
        model.train()
        for X, y in train_ld:
            opt.zero_grad()
            loss = mse(model(X.to(device)), y.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        vl = []
        with torch.no_grad():
            for X, y in val_ld:
                vl.append(mse(model(X.to(device)), y.to(device)).item())
        avg_v = np.mean(vl)
        sched.step(avg_v)
        if avg_v < best_v:
            best_v = avg_v
            best_s = {k: v.clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
        if (ep+1) % 25 == 0:
            print(f'  [{name}] Ep {ep+1}: val={avg_v:.4f}')
        if pat >= patience:
            print(f'  [{name}] Early stop ep {ep+1}')
            break
    model.load_state_dict(best_s)
    print(f'  [{name}] Done, best val={best_v:.4f}')
    return model


def train_fgginn(model, train_ld, val_ld, epochs=300, patience=50, name='FG-GINN'):
    """Обучение FG-GINN."""
    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=20, factor=0.5)
    mse = nn.MSELoss()
    best_v, best_s, pat = float('inf'), None, 0
    for ep in range(epochs):
        model.train()
        for X, yr, yg in train_ld:
            X, yr, yg = X.to(device), yr.to(device), yg.to(device)
            opt.zero_grad()
            pred, lam = model(X, yg.unsqueeze(-1))
            loss = mse(pred, yr) + 0.001 * ((lam - 0.5)**2).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        vl, vl_lam = [], []
        with torch.no_grad():
            for X, yr, yg in val_ld:
                X, yr, yg = X.to(device), yr.to(device), yg.to(device)
                pred, lam = model(X, yg.unsqueeze(-1))
                vl.append(mse(pred, yr).item())
                vl_lam.append(lam.mean().item())
        avg_v = np.mean(vl)
        sched.step(avg_v)
        if avg_v < best_v:
            best_v = avg_v
            best_s = {k: v.clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
        if (ep+1) % 25 == 0:
            print(f'  [{name}] Ep {ep+1}: val={avg_v:.4f}, λ={np.mean(vl_lam):.3f}')
        if pat >= patience:
            print(f'  [{name}] Early stop ep {ep+1}')
            break
    model.load_state_dict(best_s)
    print(f'  [{name}] Done, best val={best_v:.4f}')
    return model


def predict_lstm(model, loader, tr_mean, tr_std):
    """Предсказание LSTM → денормализованные значения."""
    model.eval()
    preds = []
    with torch.no_grad():
        for X, _ in loader:
            preds.extend(model(X.to(device)).cpu().numpy())
    return np.expm1(np.array(preds) * tr_std + tr_mean)


def predict_fgginn(model, loader, tr_mean, tr_std):
    """Предсказание FG-GINN → денормализованные значения + λ."""
    model.eval()
    preds, lambdas = [], []
    with torch.no_grad():
        for X, _, yg in loader:
            X, yg = X.to(device), yg.to(device)
            p, l = model(X, yg.unsqueeze(-1))
            preds.extend(p.cpu().numpy())
            lambdas.extend(l.cpu().numpy())
    return np.expm1(np.array(preds) * tr_std + tr_mean), np.array(lambdas)


def evaluate_all(d, lstm_preds, ginn_preds, ginn_lambdas, market_name):
    """Вычисляет метрики и печатает таблицу."""
    # Таргет — сырой gt_var (НЕ через expm1, напрямую)
    y_real = d['gt_te_raw']
    garch_raw = d['garch_te_raw']
    
    def metrics(y_true, y_pred):
        mask = ~np.isnan(y_pred)
        return {
            'MSE': mean_squared_error(y_true[mask], y_pred[mask]),
            'MAE': mean_absolute_error(y_true[mask], y_pred[mask]),
            'R2': r2_score(y_true[mask], y_pred[mask])
        }
    
    m_garch = metrics(y_real, garch_raw)
    m_lstm = metrics(y_real, lstm_preds)
    m_ginn = metrics(y_real, ginn_preds)
    
    print(f'\n{"="*60}')
    print(f'{market_name} РЕЗУЛЬТАТЫ')
    print(f'{"="*60}')
    print(f'{"Модель":<20} {"MSE":>10} {"MAE":>10} {"R²":>10}')
    print(f'{"-"*50}')
    print(f'{"GARCH(1,1)":<20} {m_garch["MSE"]:>10.2f} {m_garch["MAE"]:>10.4f} {m_garch["R2"]:>10.4f}')
    print(f'{"LSTM":<20} {m_lstm["MSE"]:>10.2f} {m_lstm["MAE"]:>10.4f} {m_lstm["R2"]:>10.4f}')
    print(f'{"FG-GINN":<20} {m_ginn["MSE"]:>10.2f} {m_ginn["MAE"]:>10.4f} {m_ginn["R2"]:>10.4f}')
    print(f'{"="*60}')
    if ginn_lambdas is not None:
        print(f'Среднее λ: {ginn_lambdas.mean():.4f} ± {ginn_lambdas.std():.4f}')
    
    return {'garch': m_garch, 'lstm': m_lstm, 'ginn': m_ginn}


def plot_results(d, lstm_preds, ginn_preds, ginn_lambdas, garch_raw, market_name):
    """Три графика: предсказания, ошибки, λ."""
    dates = d['dates_te']
    y_real = d['gt_te_raw']
    
    fig = make_subplots(rows=3, cols=1,
        subplot_titles=(f'{market_name}: Все модели vs Реальность',
                        f'{market_name}: Ошибки (Pred - Real)',
                        f'{market_name}: Адаптивный λ(t)'),
        vertical_spacing=0.1, row_heights=[0.4, 0.3, 0.3])
    
    fig.add_trace(go.Scatter(x=dates, y=y_real, name='Realized σ²',
                             line=dict(color='black', width=1.5), opacity=0.5), row=1, col=1)
    fig.add_trace(go.Scatter(x=dates, y=garch_raw, name='GARCH',
                             line=dict(color='red')), row=1, col=1)
    fig.add_trace(go.Scatter(x=dates, y=lstm_preds, name='LSTM',
                             line=dict(color='orange')), row=1, col=1)
    fig.add_trace(go.Scatter(x=dates, y=ginn_preds, name='FG-GINN',
                             line=dict(color='blue', width=2)), row=1, col=1)
    
    for name, p, color in [('GARCH', garch_raw, 'red'), ('LSTM', lstm_preds, 'orange'),
                            ('FG-GINN', ginn_preds, 'blue')]:
        fig.add_trace(go.Scatter(x=dates, y=p - y_real, name=f'{name} err',
                                 line=dict(color=color, width=1), opacity=0.6), row=2, col=1)
    fig.add_hline(y=0, line=dict(color='gray', dash='dash'), row=2, col=1)
    
    if ginn_lambdas is not None:
        fig.add_trace(go.Scatter(x=dates, y=ginn_lambdas, name='λ(t)',
                                 line=dict(color='purple'), fill='tozeroy',
                                 fillcolor='rgba(128,0,128,0.15)'), row=3, col=1)
        fig.add_hline(y=0.5, line=dict(color='green', dash='dash'), row=3, col=1)
        fig.add_hrect(y0=0, y1=0.3, fillcolor='red', opacity=0.08, line_width=0, row=3, col=1)
        fig.add_hrect(y0=0.7, y1=1.0, fillcolor='blue', opacity=0.08, line_width=0, row=3, col=1)
    
    fig.update_yaxes(title_text='σ²', row=1, col=1)
    fig.update_yaxes(title_text='Error', row=2, col=1)
    fig.update_yaxes(title_text='λ', range=[0, 1], row=3, col=1)
    fig.update_layout(title=market_name, height=1100, hovermode='x unified')
    fig.show()


def run_pipeline(returns, dates, market_name):
    """Полный pipeline: данные → обучение → метрики → графики."""
    print(f'\n{"#"*60}')
    print(f'  {market_name}')
    print(f'{"#"*60}')
    
    # Данные
    d = prepare_all_data(returns, dates)
    lstm_tr, lstm_va, lstm_te, ginn_tr, ginn_va, ginn_te = make_loaders(d)
    
    # LSTM
    print(f'\n--- LSTM ---')
    lstm = VolatilityLSTM(hidden_size=128, num_layers=3).to(device)
    lstm = train_lstm(lstm, lstm_tr, lstm_va, name=f'LSTM-{market_name}')
    lstm_preds = predict_lstm(lstm, lstm_te, d['tr_mean'], d['tr_std'])
    
    # FG-GINN
    print(f'\n--- FG-GINN ---')
    ginn = FeatureGatingGINN(hidden_size=128, num_layers=3).to(device)
    ginn = train_fgginn(ginn, ginn_tr, ginn_va, name=f'GINN-{market_name}')
    ginn_preds, ginn_lambdas = predict_fgginn(ginn, ginn_te, d['tr_mean'], d['tr_std'])
    
    # Метрики
    metrics = evaluate_all(d, lstm_preds, ginn_preds, ginn_lambdas, market_name)
    
    # Графики
    plot_results(d, lstm_preds, ginn_preds, ginn_lambdas, d['garch_te_raw'], market_name)
    
    return d, lstm, ginn, lstm_preds, ginn_preds, ginn_lambdas, metrics

## 3. Загрузка данных

In [4]:
# === IMOEX ===
imoex_df = pd.read_json('./data.json')
imoex_df.columns = ['BOARDID','SECID','TRADEDATE','SHORTNAME','NAME','CLOSE','OPEN',
                     'HIGH','LOW','VALUE','DURATION','YIELD','DECIMALS','CAPITALIZATION',
                     'CURRENCYID','DIVISOR','TRADINGSESSION','VOLUME','TRADE_SESSION_DATE',
                     'RECALC_DATE']
imoex_df['TRADEDATE'] = pd.to_datetime(imoex_df['TRADEDATE'])
imoex_df = imoex_df.sort_values('TRADEDATE').reset_index(drop=True)
imoex_df['LogReturn'] = np.log(imoex_df['CLOSE'] / imoex_df['CLOSE'].shift(1))
returns_imoex = imoex_df['LogReturn'].dropna() * 100
dates_imoex = imoex_df['TRADEDATE'].values[1:]

# === NASDAQ ===
nasdq_df = pd.read_csv('dataset/nasdq.csv')
nasdq_df['Date'] = pd.to_datetime(nasdq_df['Date'])
nasdq_df = nasdq_df.sort_values('Date').reset_index(drop=True)
nasdq_df['LogReturn'] = np.log(nasdq_df['Close'] / nasdq_df['Close'].shift(1))
returns_nasdq = nasdq_df['LogReturn'].dropna() * 100
dates_nasdq = nasdq_df['Date'].values[1:]

print(f'IMOEX: {len(returns_imoex)} доходностей')
print(f'NASDAQ: {len(returns_nasdq)} доходностей')

IMOEX: 4076 доходностей
NASDAQ: 3913 доходностей


## 4. IMOEX

In [5]:
d_imoex, lstm_imoex, ginn_imoex, lstm_p_imoex, ginn_p_imoex, ginn_l_imoex, m_imoex = run_pipeline(returns_imoex, dates_imoex, 'IMOEX')


############################################################
  IMOEX
############################################################
  GARCH rolling на 3983 точках...
    500/3983
    1000/3983
    1500/3983
    2000/3983
    2500/3983
    3000/3983
    3500/3983
  GARCH готов!
  Train: 2326, Val: 505, Test: 1062

--- LSTM ---
  [LSTM-IMOEX] Ep 25: val=0.2484
  [LSTM-IMOEX] Ep 50: val=0.1895
  [LSTM-IMOEX] Ep 75: val=0.1745
  [LSTM-IMOEX] Ep 100: val=0.1721
  [LSTM-IMOEX] Early stop ep 103
  [LSTM-IMOEX] Done, best val=0.1598

--- FG-GINN ---
  [GINN-IMOEX] Ep 25: val=0.1920, λ=0.420
  [GINN-IMOEX] Ep 50: val=0.1639, λ=0.352
  [GINN-IMOEX] Ep 75: val=0.1563, λ=0.323
  [GINN-IMOEX] Ep 100: val=0.1632, λ=0.316
  [GINN-IMOEX] Ep 125: val=0.1763, λ=0.302
  [GINN-IMOEX] Early stop ep 146
  [GINN-IMOEX] Done, best val=0.1522

IMOEX РЕЗУЛЬТАТЫ
Модель                      MSE        MAE         R²
--------------------------------------------------
GARCH(1,1)               261.23     2.3915     0

## 5. NASDAQ

In [6]:
d_nasdq, lstm_nasdq, ginn_nasdq, lstm_p_nasdq, ginn_p_nasdq, ginn_l_nasdq, m_nasdq = run_pipeline(returns_nasdq, dates_nasdq, 'NASDAQ')


############################################################
  NASDAQ
############################################################
  GARCH rolling на 3820 точках...
    500/3820
    1000/3820
    1500/3820
    2000/3820
    2500/3820
    3000/3820
    3500/3820
  GARCH готов!
  Train: 2457, Val: 529, Test: 744

--- LSTM ---
  [LSTM-NASDAQ] Ep 25: val=0.2272
  [LSTM-NASDAQ] Ep 50: val=0.3188
  [LSTM-NASDAQ] Ep 75: val=0.1661
  [LSTM-NASDAQ] Ep 100: val=0.1456
  [LSTM-NASDAQ] Ep 125: val=0.1427
  [LSTM-NASDAQ] Ep 150: val=0.1491
  [LSTM-NASDAQ] Early stop ep 152
  [LSTM-NASDAQ] Done, best val=0.1378

--- FG-GINN ---
  [GINN-NASDAQ] Ep 25: val=0.1773, λ=0.428
  [GINN-NASDAQ] Ep 50: val=0.1868, λ=0.409
  [GINN-NASDAQ] Ep 75: val=0.1468, λ=0.365
  [GINN-NASDAQ] Ep 100: val=0.1597, λ=0.302
  [GINN-NASDAQ] Ep 125: val=0.1881, λ=0.261
  [GINN-NASDAQ] Early stop ep 125
  [GINN-NASDAQ] Done, best val=0.1468

NASDAQ РЕЗУЛЬТАТЫ
Модель                      MSE        MAE         R²
---------------

## 6. Сводная таблица

In [7]:
print(f'\n{"="*70}')
print(f'{"СВОДНАЯ ТАБЛИЦА":^70}')
print(f'{"="*70}')
print(f'{"":<20} {"IMOEX R²":>12} {"NASDAQ R²":>12} {"Среднее R²":>12}')
print(f'{"-"*56}')
for model in ['garch', 'lstm', 'ginn']:
    name = {'garch': 'GARCH(1,1)', 'lstm': 'LSTM', 'ginn': 'FG-GINN'}[model]
    r2_i = m_imoex[model]['R2']
    r2_n = m_nasdq[model]['R2']
    r2_avg = (r2_i + r2_n) / 2
    print(f'{name:<20} {r2_i:>12.4f} {r2_n:>12.4f} {r2_avg:>12.4f}')
print(f'{"="*70}')


                           СВОДНАЯ ТАБЛИЦА                            
                         IMOEX R²    NASDAQ R²   Среднее R²
--------------------------------------------------------
GARCH(1,1)                 0.6418       0.0952       0.3685
LSTM                       0.5854       0.7855       0.6854
FG-GINN                    0.7703       0.7772       0.7737


## 7. Обучение на объединённых данных (IMOEX + NASDAQ)

In [8]:
X_tr_imoex_raw = d_imoex['X_tr'] * d_imoex['tr_std'] + d_imoex['tr_mean']
y_tr_imoex_raw = d_imoex['y_tr'] * d_imoex['tr_std'] + d_imoex['tr_mean']
g_tr_imoex_raw = d_imoex['g_tr'] * d_imoex['tr_std'] + d_imoex['tr_mean']

X_va_imoex_raw = d_imoex['X_va'] * d_imoex['tr_std'] + d_imoex['tr_mean']
y_va_imoex_raw = d_imoex['y_va'] * d_imoex['tr_std'] + d_imoex['tr_mean']
g_va_imoex_raw = d_imoex['g_va'] * d_imoex['tr_std'] + d_imoex['tr_mean']

X_tr_nasdaq_raw = d_nasdq['X_tr'] * d_nasdq['tr_std'] + d_nasdq['tr_mean']
y_tr_nasdaq_raw = d_nasdq['y_tr'] * d_nasdq['tr_std'] + d_nasdq['tr_mean']
g_tr_nasdaq_raw = d_nasdq['g_tr'] * d_nasdq['tr_std'] + d_nasdq['tr_mean']

X_va_nasdaq_raw = d_nasdq['X_va'] * d_nasdq['tr_std'] + d_nasdq['tr_mean']
y_va_nasdaq_raw = d_nasdq['y_va'] * d_nasdq['tr_std'] + d_nasdq['tr_mean']
g_va_nasdaq_raw = d_nasdq['g_va'] * d_nasdq['tr_std'] + d_nasdq['tr_mean']

# Шаг 2: Объединяем СЫРЫЕ (денормализованные) данные
X_tr_combined_raw = np.concatenate([X_tr_imoex_raw, X_tr_nasdaq_raw])
y_tr_combined_raw = np.concatenate([y_tr_imoex_raw, y_tr_nasdaq_raw])
X_va_combined_raw = np.concatenate([X_va_imoex_raw, X_va_nasdaq_raw])
y_va_combined_raw = np.concatenate([y_va_imoex_raw, y_va_nasdaq_raw])

g_tr_combined_raw = np.concatenate([g_tr_imoex_raw, g_tr_nasdaq_raw])
g_va_combined_raw = np.concatenate([g_va_imoex_raw, g_va_nasdaq_raw])

# Шаг 3: Нормализуем заново с ЕДИНЫМИ параметрами
combined_mean = X_tr_combined_raw.mean()
combined_std = X_tr_combined_raw.std()

X_tr_combined_norm = (X_tr_combined_raw - combined_mean) / combined_std
y_tr_combined_norm = (y_tr_combined_raw - combined_mean) / combined_std
X_va_combined_norm = (X_va_combined_raw - combined_mean) / combined_std
y_va_combined_norm = (y_va_combined_raw - combined_mean) / combined_std

g_tr_combined_norm = (g_tr_combined_raw - combined_mean) / combined_std
g_va_combined_norm = (g_va_combined_raw - combined_mean) / combined_std

print(f"Combined mean: {combined_mean:.4f}, std: {combined_std:.4f}")
print(f"Для справки — IMOEX mean: {d_imoex['tr_mean']:.4f}, NASDAQ mean: {d_nasdq['tr_mean']:.4f}")

# DataLoaders
combined_lstm_tr = DataLoader(VolatilityDataset(X_tr_combined_norm, y_tr_combined_norm), 64, shuffle=True)
combined_lstm_va = DataLoader(VolatilityDataset(X_va_combined_norm, y_va_combined_norm), 64)
combined_ginn_tr = DataLoader(GatingDataset(X_tr_combined_norm, y_tr_combined_norm, g_tr_combined_norm), 64, shuffle=True)
combined_ginn_va = DataLoader(GatingDataset(X_va_combined_norm, y_va_combined_norm, g_va_combined_norm), 64)

# Обучаем
print("\n--- LSTM (combined) ---")
lstm_combined = VolatilityLSTM(hidden_size=128, num_layers=3).to(device)
lstm_combined = train_lstm(lstm_combined, combined_lstm_tr, combined_lstm_va, name='LSTM-COMBINED')

print("\n--- FG-GINN (combined) ---")
ginn_combined = FeatureGatingGINN(hidden_size=128, num_layers=3).to(device)
ginn_combined = train_fgginn(ginn_combined, combined_ginn_tr, combined_ginn_va, name='GINN-COMBINED')

Combined mean: 0.8010, std: 0.5438
Для справки — IMOEX mean: 0.7179, NASDAQ mean: 0.8797

--- LSTM (combined) ---
  [LSTM-COMBINED] Ep 25: val=0.3069
  [LSTM-COMBINED] Ep 50: val=0.1890
  [LSTM-COMBINED] Ep 75: val=0.1547
  [LSTM-COMBINED] Ep 100: val=0.1345
  [LSTM-COMBINED] Ep 125: val=0.1405
  [LSTM-COMBINED] Early stop ep 149
  [LSTM-COMBINED] Done, best val=0.1286

--- FG-GINN (combined) ---
  [GINN-COMBINED] Ep 25: val=0.1979, λ=0.333
  [GINN-COMBINED] Ep 50: val=0.1413, λ=0.273
  [GINN-COMBINED] Ep 75: val=0.1615, λ=0.227
  [GINN-COMBINED] Ep 100: val=0.1472, λ=0.203
  [GINN-COMBINED] Early stop ep 103
  [GINN-COMBINED] Done, best val=0.1320


In [9]:
_, _, lstm_te_i, _, _, ginn_te_i = make_loaders(d_imoex)
_, _, lstm_te_n, _, _, ginn_te_n = make_loaders(d_nasdq)

# Денормализация с combined_mean/std
lstm_p_comb_i = predict_lstm(lstm_combined, lstm_te_i, combined_mean, combined_std)
ginn_p_comb_i, ginn_l_comb_i = predict_fgginn(ginn_combined, ginn_te_i, combined_mean, combined_std)

lstm_p_comb_n = predict_lstm(lstm_combined, lstm_te_n, combined_mean, combined_std)
ginn_p_comb_n, ginn_l_comb_n = predict_fgginn(ginn_combined, ginn_te_n, combined_mean, combined_std)

# Метрики
print("\n=== COMBINED модели на IMOEX ===")
evaluate_all(d_imoex, lstm_p_comb_i, ginn_p_comb_i, ginn_l_comb_i, 'IMOEX (combined)')

print("\n=== COMBINED модели на NASDAQ ===")
evaluate_all(d_nasdq, lstm_p_comb_n, ginn_p_comb_n, ginn_l_comb_n, 'NASDAQ (combined)')


=== COMBINED модели на IMOEX ===

IMOEX (combined) РЕЗУЛЬТАТЫ
Модель                      MSE        MAE         R²
--------------------------------------------------
GARCH(1,1)               261.23     2.3915     0.6418
LSTM                     561.94     2.6232     0.2294
FG-GINN                 3297.66     5.0358    -3.5224
Среднее λ: 0.2951 ± 0.2012

=== COMBINED модели на NASDAQ ===

NASDAQ (combined) РЕЗУЛЬТАТЫ
Модель                      MSE        MAE         R²
--------------------------------------------------
GARCH(1,1)                 8.85     1.5133     0.0952
LSTM                       2.09     0.4775     0.7861
FG-GINN                    2.27     0.5381     0.7679
Среднее λ: 0.2139 ± 0.1449


{'garch': {'MSE': 8.852957401995658,
  'MAE': 1.5132739387347807,
  'R2': 0.09523968545873496},
 'lstm': {'MSE': 2.092494698590767,
  'MAE': 0.4775066480539093,
  'R2': 0.7861498620510542},
 'ginn': {'MSE': 2.2714828740535142,
  'MAE': 0.5381122778835539,
  'R2': 0.7678575117575425}}

In [12]:
# === Сравнение: отдельное обучение vs combined ===
print(f'\n{"="*70}')
print(f'{"ОТДЕЛЬНО vs COMBINED":^70}')
print(f'{"="*70}')
print(f'{"Модель":<30} {"IMOEX R²":>12} {"NASDAQ R²":>12}')
print(f'{"-"*54}')

r2_i_lstm = r2_score(d_imoex['gt_te_raw'], lstm_p_imoex)
r2_n_lstm = r2_score(d_nasdq['gt_te_raw'], lstm_p_nasdq)
r2_i_lstm_c = r2_score(d_imoex['gt_te_raw'], lstm_p_comb_i)
r2_n_lstm_c = r2_score(d_nasdq['gt_te_raw'], lstm_p_comb_n)

r2_i_ginn = r2_score(d_imoex['gt_te_raw'], ginn_p_imoex)
r2_n_ginn = r2_score(d_nasdq['gt_te_raw'], ginn_p_nasdq)
r2_i_ginn_c = r2_score(d_imoex['gt_te_raw'], ginn_p_comb_i)
r2_n_ginn_c = r2_score(d_nasdq['gt_te_raw'], ginn_p_comb_n)

print(f'{"LSTM (отдельно)":<30} {r2_i_lstm:>12.4f} {r2_n_lstm:>12.4f}')
print(f'{"LSTM (combined)":<30} {r2_i_lstm_c:>12.4f} {r2_n_lstm_c:>12.4f}')
print(f'{"FG-GINN (отдельно)":<30} {r2_i_ginn:>12.4f} {r2_n_ginn:>12.4f}')
print(f'{"FG-GINN (combined)":<30} {r2_i_ginn_c:>12.4f} {r2_n_ginn_c:>12.4f}')
print(f'{"="*70}')


                         ОТДЕЛЬНО vs COMBINED                         
Модель                             IMOEX R²    NASDAQ R²
------------------------------------------------------
LSTM (отдельно)                      0.5854       0.7855
LSTM (combined)                      0.2294       0.7861
FG-GINN (отдельно)                   0.7703       0.7772
FG-GINN (combined)                  -3.5224       0.7679


In [13]:
fig_final = make_subplots(
    rows=5, cols=2,
    subplot_titles=(
        'IMOEX: Отдельные модели', 'NASDAQ: Отдельные модели',
        'IMOEX: Ошибки (Pred-Real)', 'NASDAQ: Ошибки (Pred-Real)',
        'IMOEX: Combined vs Отдельно', 'NASDAQ: Combined vs Отдельно',
        'IMOEX: Адаптивный λ(t)', 'NASDAQ: Адаптивный λ(t)',
        'IMOEX: Распределение λ', 'NASDAQ: Распределение λ'
    ),
    vertical_spacing=0.06,
    horizontal_spacing=0.1,
    row_heights=[0.25, 0.15, 0.25, 0.2, 0.15]
)

colors = {'GARCH': 'red', 'LSTM': 'orange', 'GINN': 'blue'}

# ========== Row 1-2: Отдельные модели ==========
for col, (name, d, lp, gp, gl, garch) in enumerate([
    ('IMOEX', d_imoex, lstm_p_imoex, ginn_p_imoex, ginn_l_imoex, d_imoex['garch_te_raw']),
    ('NASDAQ', d_nasdq, lstm_p_nasdq, ginn_p_nasdq, ginn_l_nasdq, d_nasdq['garch_te_raw'])
], 1):
    y_real = d['gt_te_raw']
    dates = d['dates_te']
    
    # Row 1: Predictions (separate models)
    fig_final.add_trace(
        go.Scatter(x=dates, y=y_real, name=f'{name} Real', 
                   line=dict(color='black', width=1.2), opacity=0.4, showlegend=(col==1)),
        row=1, col=col
    )
    for model_name, preds, color, dash in [
        ('GARCH', garch, 'red', None), 
        ('LSTM', lp, 'orange', None), 
        ('GINN', gp, 'blue', None)
    ]:
        fig_final.add_trace(
            go.Scatter(x=dates, y=preds, name=f'{name} {model_name}',
                       line=dict(color=color, width=1.5, dash=dash), showlegend=(col==1)),
            row=1, col=col
        )
    
    # Row 2: Errors
    for model_name, preds, color in [
        ('GARCH', garch, 'red'), ('LSTM', lp, 'orange'), ('GINN', gp, 'blue')
    ]:
        err = preds - y_real
        fig_final.add_trace(
            go.Scatter(x=dates, y=err, name=f'{name} {model_name} err',
                       line=dict(color=color, width=0.8), opacity=0.5, showlegend=(col==1)),
            row=2, col=col
        )
    fig_final.add_hline(y=0, line=dict(color='gray', dash='dash', width=1), row=2, col=col)
    
    # Текст с метриками
    r2_garch = r2_score(y_real, garch)
    r2_lstm = r2_score(y_real, lp)
    r2_ginn = r2_score(y_real, gp)
    fig_final.add_annotation(
        xref=f'x{col}', yref=f'y{col}',
        x=dates[len(dates)//2], y=y_real.max()*0.9,
        text=f'GARCH R²={r2_garch:.3f}<br>LSTM R²={r2_lstm:.3f}<br>GINN R²={r2_ginn:.3f}',
        showarrow=False, font=dict(size=10), bgcolor='rgba(255,255,255,0.8)',
        row=1, col=col
    )

# ========== Row 3: Combined vs Отдельно (GINN) ==========
for col, (name, d, gp_sep, gp_comb) in enumerate([
    ('IMOEX', d_imoex, ginn_p_imoex, ginn_p_comb_i),
    ('NASDAQ', d_nasdq, ginn_p_nasdq, ginn_p_comb_n)
], 1):
    y_real = d['gt_te_raw']
    dates = d['dates_te']
    
    fig_final.add_trace(
        go.Scatter(x=dates, y=y_real, name=f'{name} Real',
                   line=dict(color='black', width=1), opacity=0.3, showlegend=False),
        row=3, col=col
    )
    fig_final.add_trace(
        go.Scatter(x=dates, y=gp_sep, name=f'{name} GINN sep',
                   line=dict(color='blue', width=2), showlegend=(col==1)),
        row=3, col=col
    )
    fig_final.add_trace(
        go.Scatter(x=dates, y=gp_comb, name=f'{name} GINN comb',
                   line=dict(color='cyan', width=2, dash='dash'), showlegend=(col==1)),
        row=3, col=col
    )
    
    # Метрики combined
    r2_sep = r2_score(y_real, gp_sep)
    r2_comb = r2_score(y_real, gp_comb)
    fig_final.add_annotation(
        xref=f'x{col}', yref=f'y{col}',
        x=dates[len(dates)//2], y=y_real.max()*0.9,
        text=f'Отдельно R²={r2_sep:.3f}<br>Combined R²={r2_comb:.3f}',
        showarrow=False, font=dict(size=10), bgcolor='rgba(255,255,255,0.8)',
        row=3, col=col
    )

# ========== Row 4-5: Lambda (только separate GINN) ==========
for col, (name, gl) in enumerate([
    ('IMOEX', ginn_l_imoex), ('NASDAQ', ginn_l_nasdq)
], 1):
    dates = d_imoex['dates_te'] if name == 'IMOEX' else d_nasdq['dates_te']
    
    # Row 4: Lambda time series
    fig_final.add_trace(
        go.Scatter(x=dates, y=gl, name=f'{name} λ(t)',
                   line=dict(color='purple', width=1.2),
                   fill='tozeroy', fillcolor='rgba(128,0,128,0.12)', showlegend=(col==1)),
        row=4, col=col
    )
    fig_final.add_hline(y=0.5, line=dict(color='green', dash='dash', width=1.2), row=4, col=col)
    fig_final.add_hrect(y0=0, y1=0.3, fillcolor='red', opacity=0.06, line_width=0, row=4, col=col)
    fig_final.add_hrect(y0=0.7, y1=1.0, fillcolor='blue', opacity=0.06, line_width=0, row=4, col=col)
    fig_final.add_annotation(text='GARCH zone', xref=f'x{col}', yref=f'y{col}',
                             x=0.02, y=0.15, showarrow=False, font=dict(size=9, color='red'), row=4, col=col)
    fig_final.add_annotation(text='Data zone', xref=f'x{col}', yref=f'y{col}',
                             x=0.02, y=0.85, showarrow=False, font=dict(size=9, color='blue'), row=4, col=col)
    
    # Row 5: Lambda histogram
    fig_final.add_trace(
        go.Histogram(x=gl, nbinsx=35, name=f'{name} λ dist',
                     marker_color='purple', opacity=0.6, showlegend=False),
        row=5, col=col
    )
    fig_final.add_vline(x=np.mean(gl), line=dict(color='purple', width=2, dash='dash'),
                        annotation_text=f'μ={np.mean(gl):.3f}', row=5, col=col)
    fig_final.add_vline(x=0.5, line=dict(color='green', width=1.2, dash='dot'), row=5, col=col)

# ========== Настройка осей ==========
for row in [1, 3]:
    for col in [1, 2]:
        fig_final.update_yaxes(title_text='σ² variance', row=row, col=col)
fig_final.update_yaxes(title_text='Error', row=2, col=1)
fig_final.update_yaxes(title_text='Error', row=2, col=2)
fig_final.update_yaxes(title_text='λ', range=[0, 1], row=4, col=1)
fig_final.update_yaxes(title_text='λ', range=[0, 1], row=4, col=2)
fig_final.update_yaxes(title_text='Count', row=5, col=1)
fig_final.update_yaxes(title_text='Count', row=5, col=2)
fig_final.update_xaxes(title_text='λ', row=5, col=1)
fig_final.update_xaxes(title_text='λ', row=5, col=2)

fig_final.update_layout(
    title='Feature Gating GINN: полное сравнение IMOEX vs NASDAQ',
    height=1600,
    hovermode='x unified',
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5, font=dict(size=9))
)

fig_final.show()
